# 04. Race conditions

**Tier:** Deep. **Why you'd use this:** Two threads update the same counter with `+=` and you lose increments. A lock makes the read–modify–write step exclusive.

**After this lesson you can:** explain a lost update from unlocked `+=`; make the race visible with read–sleep–write; protect the counter with a `Lock`.
About 30 minutes. **Uses:** [Threads](01-threads-lesson.ipynb)


## Quick review

1. What does `await` do to the current task?
2. Why avoid `asyncio.run` inside a notebook cell that already has a loop?

<details><summary>Answers</summary>

It pauses that task so others can run until the wait finishes. The notebook already has a running loop — nest or use top-level await instead.
</details>


## The idea

A race condition happens when two threads interleave operations on shared data and the result depends on timing. `counter += 1` is not atomic: it reads, adds, and writes. If both threads read before either writes, one update is lost. A `Lock` makes the critical section run without interleaving. Sleeping between read and write makes the bug show up every run.

Critical sections should be as short as you can make them while still covering every step that must look atomic. Queues move ownership of data so fewer structures need locks. Immutable messages reduce races. Tests that only pass sometimes are a smell: force the interleaving in a lesson, then lock it.

<details><summary>Explain more</summary>

The GIL does not make `+=` safe. It can switch threads between bytecode instructions. Free-threaded builds make races easier to hit; locks still matter.
</details>


## See it work

Same bump under a lock.


In [1]:
import threading
import time

counter = 0
lock = threading.Lock()


def bump():
    global counter
    # 1. Hold the lock for the whole read-modify-write.
    with lock:
        value = counter
        time.sleep(0.01)
        counter = value + 1


threads = [threading.Thread(target=bump) for _ in range(2)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print(counter)

2


## Mental model

A race is two cooks editing one recipe card. Each reads "salt: 1", adds one, writes "salt: 2", but if both read before either writes, dinner gets one pinch instead of two. Locks are a kitchen timer that only one cook may hold. Sleep between read and write is a teaching cheat that makes the clash inevitable. Production code should not need that sleep; tests might use barriers or the same pattern to prove a fix.

| pattern | safe? | notes |
|---|---|---|
| local variable only | yes | not shared |
| append-only under GIL assumptions | still risky | use a lock or queue |
| `counter += 1` unlocked | no | classic lost update |
| `with lock: counter += 1` | yes | serialises RMW |
| queue between threads | usually | ownership transfers |

Prefer designing shared state out. When you cannot, name the lock after the data it protects and document which operations need it. Deadlocks come from taking two locks in different orders—keep a single lock or a fixed order when you grow beyond this lesson.

In reviews, ask "who else can see this object?" If the answer is more than one thread, name the lock or change the design. Atomic operations in other languages still need care in Python because `+=` is not a single opcode here.


## Check yourself

Answer these from the example above before you run the cell.


1. Which three steps make `+=` unsafe?
2. Why does `sleep` between read and write make the race reliable?
3. What does `with lock` guarantee if the body raises?

<details><summary>Answers</summary>

1. Read, add, write. 2. It yields so the other thread reads the old value. 3. The lock is released anyway.
</details>


In [2]:
import threading
import time

counter = 0


def bump():
    global counter
    value = counter
    time.sleep(0.01)
    counter = value + 1


threads = [threading.Thread(target=bump) for _ in range(2)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print(counter)

1


<details><summary>Check your answers</summary>

`1`, not `2`. Both threads read `0`, sleep, then write `1`.
</details>


## Why it works this way

`counter += 1` compiles to several bytecode steps: load global, load constant, add, store global. Between those steps another thread can run. The GIL serializes bytecode execution but still releases between instructions, so interleaving remains possible. Inserting `sleep` after the read forces the scheduler to run the other thread while both still hold the old value, which turns a heisenbug into a reliable demo. Deep dive: [The GIL and free-threading](../../internals/12-gil/01-gil-lesson.ipynb).

A `Lock` allows only one thread into the critical section. `with lock:` acquires and releases even if the body errors. Cost is contention: threads wait for the lock. Keep critical sections small. The design reason for explicit locks is correctness of shared mutable state; the GIL is not that lock. Free-threaded builds make lost updates more common without changing the need for a lock.

Trace of the unlocked race:

| step | thread A | thread B | counter |
|---|---|---|---|
| 1 | read 0 | | 0 |
| 2 | sleep | read 0 | 0 |
| 3 | write 1 | sleep | 1 |
| 4 | | write 1 | 1 |

With a lock, step 2 cannot read until A finishes the write. Evidence: unlocked print `1`, locked print `2`. A high-iteration unlocked loop often prints less than the expected total; the locked version prints exactly.

Race demos that use `sleep` to force interleaving can still vary under load on a busy host. Cells that only print the contested total are tagged `volatile` when the thread schedule is not guaranteed. Treat those prints as evidence of the pattern, not as a fixed microbenchmark.


In [3]:
import threading
import time


def race(locked):
    counter = {"value": 0}
    lock = threading.Lock()

    def bump():
        if locked:
            with lock:
                value = counter["value"]
                time.sleep(0.01)
                counter["value"] = value + 1
        else:
            value = counter["value"]
            time.sleep(0.01)
            counter["value"] = value + 1

    threads = [threading.Thread(target=bump) for _ in range(2)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    return counter["value"]


print(race(False))
print(race(True))

1
2


## When to use it

Use a lock around every read-modify-write on shared mutable data. Prefer queues or process isolation when you can avoid sharing. Don't sleep in production critical sections except in demos. Don't assume the GIL makes `+=` safe.

Lock every read-modify-write on shared mutable data, or redesign so only one thread owns the data. Prefer `queue.Queue` for producer/consumer. Prefer processes when isolation is simpler than locking. Never use sleep in production critical sections except when reproducing a race in a test or lesson.


## Common mistake

Updating a shared counter from threads without a lock and treating an occasional correct total as proof of safety.


In [4]:
import threading

counter = 0


def bump():
    global counter
    for _ in range(10_000):
        counter += 1


threads = [threading.Thread(target=bump) for _ in range(4)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print(counter)

40000


The print is often less than `40000`. Lost updates are silent. Guard the `+=` with a lock (or use a queue of increments).


In [5]:
import threading

counter = 0
lock = threading.Lock()


def bump():
    global counter
    for _ in range(10_000):
        with lock:
            counter += 1


threads = [threading.Thread(target=bump) for _ in range(4)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print(counter)

40000


## Try it

Change the locked demo to bump five threads once each. What total should print, and what goes wrong if you lock only the write?


## Recap

- `+=` on shared data is a race without a lock.
- Read–sleep–write demos make the lost update deterministic.
- `Lock` serializes the critical section.

**Next:** [practice](04-race-conditions-practice.ipynb)
